# CS F425 Deep Learning Project
## QLoRA Fine-Tuning of Phi-2 for Structured Data Analysis

Fine-tunes `microsoft/phi-2` (4-bit QLoRA) to act as an AI agent over `sales_data.csv`.  
Given a natural language query, the model outputs:
```json
{"actions": ["filter_data(column='year', value=2022)", "aggregate_sum(column='revenue')"], "answer": 52345678.12}
```

**Runtime target**: ≤6 hours on Colab free-tier T4 GPU.

## Cell 1 — Install Packages

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

DEPS_SENTINEL = Path("/tmp/.csf425_phi2_deps_ready")
PACKAGES = [
    "transformers>=4.41.0",
    "peft>=0.10.0",
    "trl>=0.9.0",
    "accelerate>=0.29.3",
    "datasets>=2.19.0",
    "bitsandbytes>=0.44.0",
    "einops",
]

if not DEPS_SENTINEL.exists():
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *PACKAGES])
    DEPS_SENTINEL.write_text("ok")
    print("Dependencies installed or upgraded. Restarting the Colab runtime once...")
    os.kill(os.getpid(), 9)

print("Dependencies already prepared for this runtime.")


In [ ]:
# Cell 1 restarts the Colab runtime once after upgrading packages.
# After the reconnect, continue from the next cell.


## Cell 2 — Imports & GPU Check

In [ ]:
import json
import random
import re

import pandas as pd
import torch
from datasets import Dataset
from packaging import version
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)

try:
    import trl
except Exception as exc:
    raise RuntimeError(
        "Failed to import trl. Re-run Cell 1 and let Colab restart once."
    ) from exc

if not hasattr(trl, "SFTTrainer"):
    raise RuntimeError(
        "trl.SFTTrainer is unavailable. Re-run Cell 1 and let Colab restart once."
    )

SFTTrainer = trl.SFTTrainer
SFTConfig = getattr(trl, "SFTConfig", TrainingArguments)
_USE_SFTCONFIG = hasattr(trl, "SFTConfig")

import transformers as _tf

if version.parse(_tf.__version__) < version.parse("4.41.0"):
    raise RuntimeError(
        f"transformers=={_tf.__version__} is too old for this notebook. "
        "Re-run Cell 1 and let Colab restart once."
    )

print(f"trl         : {trl.__version__}  (SFTConfig available: {_USE_SFTCONFIG})")
print(f"transformers: {_tf.__version__}")

assert torch.cuda.is_available(), "No GPU. In Colab, switch to a T4 GPU runtime."
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## Cell 3 — Mount Google Drive

Saves adapter weights and checkpoints to Drive so they persist across Colab sessions.

**One-time setup**: Upload these files to `MyDrive/CS_F425_Project/`:
- `sales_data.csv`
- `agent_trajectories_2k.json`
- `tool_executor.py`
- `run_pipeline.py`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os
DRIVE_DIR = "/content/drive/MyDrive/CS_F425_Project"
ADAPTER_DIR = f"{DRIVE_DIR}/phi2-agent-adapter"
CKPT_DIR = f"{DRIVE_DIR}/phi2-agent-qlora"

os.makedirs(ADAPTER_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

# Make tool_executor.py importable
sys.path.insert(0, DRIVE_DIR)

print(f"Drive dir: {DRIVE_DIR}")
print(f"Contents: {os.listdir(DRIVE_DIR)}")

## Cell 4 — Load Data

In [ ]:
df = pd.read_csv(f"{DRIVE_DIR}/sales_data.csv")
print(f"sales_data shape: {df.shape}")
print(df.dtypes)
print(df.head(3))

with open(f"{DRIVE_DIR}/agent_trajectories_2k.json") as f:
    trajectories = json.load(f)
print(f"\nTrajectories loaded: {len(trajectories)}")
print("Sample entry:", json.dumps(trajectories[0], indent=2))

## Cell 5 — Pre-compute Answers

The training data has `query` + `actions` but **no answers**.  
We execute each action sequence against `sales_data.csv` using `ToolExecutor` to obtain the gold answer.

We also extend `parse_agent_action` to handle `aggregate_mean`, `aggregate_count`, and string filter values that the original `run_pipeline.py` misses.

In [ ]:
from numbers import Integral, Real

from tool_executor import ToolExecutor


def parse_agent_action(action_str):
    """Parse one string-form action into ToolExecutor format."""
    if action_str.startswith("filter_data"):
        m = re.search(r"column='([^']+)', value=(\d+)", action_str)
        if m:
            return {
                "tool": "filter",
                "args": {"column": m.group(1), "op": "==", "value": int(m.group(2))},
            }

        m = re.search(r"column='([^']+)', value='([^']+)'", action_str)
        if m:
            return {
                "tool": "filter",
                "args": {"column": m.group(1), "op": "==", "value": m.group(2)},
            }

    elif action_str.startswith("group_by"):
        m = re.search(r"column='([^']+)'", action_str)
        if m:
            return {"tool": "groupby", "args": {"column": m.group(1)}}

    elif action_str.startswith("aggregate_sum"):
        m = re.search(r"column='([^']+)'", action_str)
        if m:
            return {"tool": "aggregate", "args": {"column": m.group(1), "agg": "sum"}}

    elif action_str.startswith("aggregate_mean"):
        m = re.search(r"column='([^']+)'", action_str)
        if m:
            return {"tool": "aggregate", "args": {"column": m.group(1), "agg": "mean"}}

    elif action_str.startswith("aggregate_count"):
        m = re.search(r"column='([^']+)'", action_str)
        if m:
            return {"tool": "aggregate", "args": {"column": m.group(1), "agg": "count"}}

    elif action_str.startswith("sort_by"):
        m = re.search(r"column='([^']+)', order='([^']+)'", action_str)
        if m:
            return {
                "tool": "sort",
                "args": {"column": m.group(1), "ascending": m.group(2) != "desc"},
            }

    elif action_str.startswith("top_k"):
        m = re.search(r"k=(\d+)", action_str)
        if m:
            return {"tool": "topk", "args": {"k": int(m.group(1))}}

    return None


def clean_scalar(value):
    if hasattr(value, "item"):
        try:
            value = value.item()
        except Exception:
            pass

    if isinstance(value, bool):
        return bool(value)
    if isinstance(value, Integral):
        return int(value)
    if isinstance(value, Real):
        return round(float(value), 4)
    return value


def result_to_python(actions, result_df):
    """Convert ToolExecutor output into the JSON shape used for supervision."""
    if result_df is None or (hasattr(result_df, "empty") and result_df.empty):
        return None

    action_names = [action.split("(")[0] for action in actions]

    if result_df.shape == (1, 1):
        return clean_scalar(result_df.iloc[0, 0])

    if action_names == ["group_by", "aggregate_mean"] and result_df.shape[1] == 2:
        key_col, value_col = result_df.columns
        return {
            str(row[key_col]): clean_scalar(row[value_col])
            for _, row in result_df.iterrows()
        }

    return [
        {str(key): clean_scalar(value) for key, value in row.items()}
        for row in result_df.to_dict(orient="records")
    ]


def compute_answer(actions, df):
    """Parse and execute a list of action strings."""
    parsed = []
    for action in actions:
        try:
            parsed_action = parse_agent_action(action)
        except Exception:
            parsed_action = None

        if parsed_action is not None:
            parsed.append(parsed_action)

    if not parsed:
        return None

    try:
        result = ToolExecutor(df.copy()).execute(parsed)
        return result_to_python(actions, result)
    except Exception:
        return None


training_data = []
skipped = 0

for entry in trajectories:
    answer = compute_answer(entry["actions"], df)
    if answer is None:
        skipped += 1
        continue

    output_json = json.dumps(
        {"actions": entry["actions"], "answer": answer},
        ensure_ascii=False,
    )
    training_data.append({"query": entry["query"], "output_json": output_json})

print(f"Valid examples : {len(training_data)} / {len(trajectories)}")
print(f"Skipped        : {skipped}")
print("\nSample:")
print(json.dumps(json.loads(training_data[0]["output_json"]), indent=2, ensure_ascii=False))


## Cell 6 — Schema String & Prompt Template

The prompt follows the same `### Task / ### Schema / ### Question / ### Answer` pattern as Lab02.  
The schema string is fixed — it is identical at training and inference time.

In [ ]:
SCHEMA = """Table: sales_data
Columns: date (date), year (int), month (int), city (str), region (str),
         product (str), category (str), revenue (float), units_sold (int),
         cost (float), profit (float)

Available actions (use exactly this syntax):
  filter_data(column='col', value=val)
  group_by(column='col')
  aggregate_sum(column='col')
  aggregate_mean(column='col')
  aggregate_count(column='col')
  sort_by(column='col', order='asc'|'desc')
  top_k(k=N)"""

PROMPT_TEMPLATE = """### Task
Analyze the sales data and answer the query.
Output ONLY a JSON object with an \"actions\" list and an \"answer\" field. No other text.

### Schema
{schema}

### Question
{question}

### Answer
"""

MAX_SEQ_LENGTH = 384


def make_prompt(question: str) -> str:
    return PROMPT_TEMPLATE.format(schema=SCHEMA, question=question)


EOS = None


def format_example(row):
    return {"text": make_prompt(row["query"]) + row["output_json"] + EOS}


print(make_prompt(training_data[0]["query"]))
print(f"Configured max sequence length: {MAX_SEQ_LENGTH}")


## Cell 7 — Split Dataset

In [ ]:
random.seed(42)
random.shuffle(training_data)

TRAIN_SIZE = min(1800, int(len(training_data) * 0.9))
VALID_SIZE = 200

raw_train = training_data[:TRAIN_SIZE]
raw_valid = training_data[TRAIN_SIZE : TRAIN_SIZE + VALID_SIZE]

print(f"Train : {len(raw_train)}")
print(f"Valid : {len(raw_valid)}")

## Cell 8 — Load Tokenizer

In [ ]:
MODEL_NAME = "microsoft/phi-2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
EOS = tokenizer.eos_token

print(f"EOS token: {EOS!r}")

sample_text = make_prompt(raw_train[0]["query"]) + raw_train[0]["output_json"] + EOS
n_tokens = len(tokenizer(sample_text).input_ids)
print(f"Sample token length: {n_tokens} (target <= {MAX_SEQ_LENGTH})")


## Cell 9 — Build HuggingFace Datasets

In [ ]:
def fmt(row):
    return format_example(row)  # EOS is now set

train_ds = Dataset.from_list(raw_train).map(
    fmt, remove_columns=["query", "output_json"]
)
valid_ds = Dataset.from_list(raw_valid).map(
    fmt, remove_columns=["query", "output_json"]
)

print(f"Train dataset : {len(train_ds)} examples")
print(f"Valid dataset : {len(valid_ds)} examples")
print("\nSample text (truncated):")
print(train_ds[0]["text"][:500])

## Cell 10 — Load Phi-2 in 4-bit (QLoRA)

Config mirrors Lab02 exactly: NF4 double-quant, bfloat16 compute dtype, gradient checkpointing.

In [ ]:
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f"Compute dtype: {compute_dtype}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

!nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader


## Cell 11 — Apply LoRA Adapters

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "dense", "fc1", "fc2"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected: ~0.84% trainable (~23M of 2.8B params)

## Cell 12 — Fine-Tune with SFTTrainer

Config mirrors Lab02: cosine LR, paged AdamW-8bit, effective batch 16, 2 epochs.  
Checkpoints saved to Drive so training can resume on Colab disconnect.

In [ ]:
import inspect as _inspect

_sft_sig = set(_inspect.signature(SFTConfig.__init__).parameters.keys())
_trainer_sig = set(_inspect.signature(SFTTrainer.__init__).parameters.keys())

_eval_key = "eval_strategy" if "eval_strategy" in _sft_sig else "evaluation_strategy"
_tok_key = "processing_class" if "processing_class" in _trainer_sig else "tokenizer"

_steps_per_epoch = max(1, len(train_ds) // (4 * 4))
_warmup_steps = max(1, int(0.05 * _steps_per_epoch * 2))

_cfg_extra = {}
_trainer_extra = {}
for _param, _val in [
    ("max_seq_length", MAX_SEQ_LENGTH),
    ("dataset_text_field", "text"),
    ("packing", False),
]:
    if _param in _sft_sig:
        _cfg_extra[_param] = _val
    elif _param in _trainer_sig:
        _trainer_extra[_param] = _val

_common_args = dict(
    output_dir=CKPT_DIR,
    seed=42,
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=_warmup_steps,
    optim="paged_adamw_8bit",
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    logging_steps=50,
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
)

training_args = SFTConfig(
    **_common_args,
    **{_eval_key: "epoch"},
    **_cfg_extra,
)

trainer = SFTTrainer(
    model=model,
    **{_tok_key: tokenizer},
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    **_trainer_extra,
)

print(f"Training on {len(train_ds)} examples, validating on {len(valid_ds)}")
print(f"Epochs: 2  |  Effective batch: 16  |  Warmup steps: {_warmup_steps}")
print(f"Max sequence length: {MAX_SEQ_LENGTH}")
print(f"SFT params -> SFTConfig: {_cfg_extra}  |  SFTTrainer: {_trainer_extra}")
print(f"Tokenizer key: {_tok_key!r}")
trainer.train()


## Cell 13 — Save Adapter Weights to Drive

In [ ]:
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print(f"Adapter saved to: {ADAPTER_DIR}")
print("Files:")
for fname in sorted(os.listdir(ADAPTER_DIR)):
    size_mb = os.path.getsize(f"{ADAPTER_DIR}/{fname}") / 1e6
    print(f"  {fname:<40s}  {size_mb:.2f} MB")

## Cell 14 — Inference Function

Given a natural language question, generates the JSON output and parses it back to a Python dict.

In [ ]:
def generate_answer(question: str, max_new_tokens: int = 200) -> dict:
    """Run the fine-tuned model on a question and return a parsed JSON dict."""
    model.eval()
    model.config.use_cache = True

    prompt = make_prompt(question)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_tokens = out[0][inputs["input_ids"].shape[1] :]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", raw, re.DOTALL)
        if match:
            try:
                return json.loads(match.group())
            except json.JSONDecodeError:
                pass
        return {"raw_output": raw}


## Cell 15 — Evaluation on Held-Out Queries

Runs the model on a set of test questions and cross-checks the model's answer against
the ground-truth answer computed directly by `ToolExecutor`.

In [ ]:
def normalize_answer(value):
    if isinstance(value, float):
        return round(value, 4)
    if isinstance(value, list):
        return [normalize_answer(item) for item in value]
    if isinstance(value, dict):
        return {key: normalize_answer(val) for key, val in value.items()}
    return value


test_queries = [
    "What is the total revenue for 2022?",
    "Which city had the highest profit in 2021? Top 1",
    "What is the average revenue by city?",
    "List top 3 cities by revenue in 2022.",
    "What is the total profit for 2023?",
]

for q in test_queries:
    result = generate_answer(q)
    actions = result.get("actions", [])
    model_answer = result.get("answer", "N/A")
    verified_answer = compute_answer(actions, df) if actions else None

    print(f"Q: {q}")
    print(f"  Actions          : {actions}")
    print(f"  Model answer     : {model_answer}")
    print(f"  Verified answer  : {verified_answer}")
    print(f"  Match            : {normalize_answer(model_answer) == normalize_answer(verified_answer)}")
    print()
